# ==============================================================================
# AegisStep-28M: Open-Weight Sparse MoE Process Reward Model
# ==============================================================================


In [1]:
import os
import sys
import math
import time
import json
import zlib
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, balanced_accuracy_score


# Install required Hugging Face ecosystem packages - if not present
os.system("pip install -q datasets transformers huggingface_hub scikit-learn")

from datasets import load_dataset
from transformers import AutoTokenizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from huggingface_hub import HfApi, login


## 1. REPRODUCIBILITY & DEVICE SETUP

In [2]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Execution Device: {DEVICE}")

[*] Execution Device: cuda


## 2. REAL DATASET PIPELINE (Math-Shepherd) & TOKENIZER SETUP

In [3]:
SPECIAL_TOKENS = {"additional_special_tokens": ["<step>", "<prm_start>", "<prm_end>"]}

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.add_special_tokens(SPECIAL_TOKENS)
tokenizer.pad_token = tokenizer.eos_token
STEP_TOKEN_ID = tokenizer.convert_tokens_to_ids("<step>")

_RAW_CACHE = None

def get_raw_dataset():
    global _RAW_CACHE
    if _RAW_CACHE is None:
        print("[*] Loading peiyi9979/Math-Shepherd (one time)...")
        _RAW_CACHE = load_dataset("peiyi9979/Math-Shepherd", split="train").shuffle(seed=42)
    return _RAW_CACHE


class MathShepherdDataset(Dataset):
    def __init__(self, tokenizer, mode="train", max_items=None, max_seq_len=512):
        assert mode in ("train", "val", "test")
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.samples = []

        print(f"[*] Building '{mode}' split...")
        for item in get_raw_dataset():
            raw_input = item.get("input", "")
            raw_label = item.get("label", "")
            if not isinstance(raw_label, str):
                continue

            question = raw_input.split("Step 1:")[0]
            bucket = zlib.crc32(question.encode("utf-8")) % 100
            item_mode = "train" if bucket < 90 else ("val" if bucket < 95 else "test")
            if item_mode != mode:
                continue

            segments = raw_input.split("ки")
            if len(segments) < 2:
                continue

            step_labels, pos, valid = [], 0, True
            for seg in segments[:-1]:
                pos += len(seg)
                if pos >= len(raw_label) or raw_label[pos] not in "+-":
                    valid = False
                    break
                step_labels.append(1.0 if raw_label[pos] == "+" else 0.0)
                pos += 1
            if not valid or not step_labels:
                continue

            formatted_text = f"<prm_start> {raw_input.replace('ки', '<step>')} <prm_end>"
            self.samples.append({"text": formatted_text, "labels": step_labels})

            if max_items and len(self.samples) >= max_items:
                break

        all_labels = [l for s in self.samples for l in s["labels"]]
        pos_rate = float(np.mean(all_labels)) if all_labels else 0.0
        print(f"[+] {mode}: {len(self.samples)} chains | positive-step rate: {pos_rate*100:.1f}%")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        enc = self.tokenizer(
            item["text"], truncation=True,
            max_length=self.max_seq_len, return_tensors="pt"   # padding done in collate_fn
        )
        input_ids = enc["input_ids"].squeeze(0)
        labels = torch.full(input_ids.shape, -100.0, dtype=torch.float32)

        step_positions = (input_ids == STEP_TOKEN_ID).nonzero(as_tuple=True)[0]
        n = min(len(step_positions), len(item["labels"]))
        for i in range(n):
            labels[step_positions[i]] = item["labels"][i]

        return {"input_ids": input_ids, "step_labels": labels}


def collate_fn(batch):
    max_len = max(b["input_ids"].size(0) for b in batch)
    ids = torch.full((len(batch), max_len), tokenizer.pad_token_id, dtype=torch.long)
    lab = torch.full((len(batch), max_len), -100.0, dtype=torch.float32)
    for i, b in enumerate(batch):
        n = b["input_ids"].size(0)
        ids[i, :n] = b["input_ids"]
        lab[i, :n] = b["step_labels"]
    return {"input_ids": ids, "step_labels": lab}

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## 3. AEGISSTEP-28M ARCHITECTURE FROM SCRATCH

In [4]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        var = torch.mean(x ** 2, dim=-1, keepdim=True)
        return x * torch.rsqrt(var + self.eps) * self.weight

class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, max_seq_len: int = 512, base: float = 10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        t = torch.arange(max_seq_len, dtype=torch.float32)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)

    def forward(self, seq_len: int):
        return self.cos_cached[:seq_len, :], self.sin_cached[:seq_len, :]

def rotate_half(x):
    x1 = x[..., :x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(2)
    sin = sin.unsqueeze(0).unsqueeze(2)
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, n_kv_heads: int, head_dim: int):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = head_dim
        self.num_queries_per_kv = n_heads // n_kv_heads

        self.q_proj = nn.Linear(d_model, n_heads * head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * head_dim, bias=False)
        self.o_proj = nn.Linear(n_heads * head_dim, d_model, bias=False)

    def forward(self, x, cos, sin):
        B, S, _ = x.shape
        q = self.q_proj(x).view(B, S, self.n_heads, self.head_dim)
        k = self.k_proj(x).view(B, S, self.n_kv_heads, self.head_dim)
        v = self.v_proj(x).view(B, S, self.n_kv_heads, self.head_dim)

        q, k = apply_rotary_pos_emb(q, k, cos, sin)

        if self.num_queries_per_kv > 1:
            k = k.repeat_interleave(self.num_queries_per_kv, dim=2)
            v = v.repeat_interleave(self.num_queries_per_kv, dim=2)

        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        attn_out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, S, -1)
        return self.o_proj(attn_out)

class SwiGLUExpert(nn.Module):
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)
        self.w3 = nn.Linear(d_model, d_ff, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

class SparseMoELayer(nn.Module):
    def __init__(self, d_model: int, d_ff: int, num_experts: int = 4, top_k: int = 2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.gate = nn.Linear(d_model, num_experts, bias=False)
        self.experts = nn.ModuleList([SwiGLUExpert(d_model, d_ff) for _ in range(num_experts)])

    def forward(self, x):
        B, S, D = x.shape
        x_flat = x.view(-1, D)
        router_logits = self.gate(x_flat)

        probs = F.softmax(router_logits, dim=-1)
        weights, indices = torch.topk(probs, self.top_k, dim=-1)
        weights = weights / weights.sum(dim=-1, keepdim=True)

        # Auxiliary Load Balancing Loss computation
        expert_mask = F.one_hot(indices, num_classes=self.num_experts).sum(dim=1).float()
        f_i = expert_mask.mean(dim=0)
        P_i = probs.mean(dim=0)
        aux_loss = self.num_experts * torch.sum(f_i * P_i)

        final_output = torch.zeros_like(x_flat)
        for i, expert in enumerate(self.experts):
            expert_mask_i = (indices == i)
            token_idx, top_k_idx = torch.where(expert_mask_i)
            if token_idx.numel() > 0:
                in_tokens = x_flat[token_idx]
                out_tokens = expert(in_tokens)
                gate_weights = weights[token_idx, top_k_idx].unsqueeze(-1)
                final_output.index_add_(0, token_idx, out_tokens * gate_weights)

        return final_output.view(B, S, D), aux_loss

class AegisStep28M(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 288,
        n_layers: int = 6,
        n_heads: int = 8,
        n_kv_heads: int = 2,
        d_ff: int = 576,
        num_experts: int = 4,
        top_k: int = 2,
        max_seq_len: int = 512
    ):
        super().__init__()
        self.tok_embeddings = nn.Embedding(vocab_size, d_model)
        self.rope = RotaryEmbedding(dim=d_model // n_heads, max_seq_len=max_seq_len)
        self.drop = nn.Dropout(0.1)

        self.layers = nn.ModuleList([])
        for _ in range(n_layers):
            self.layers.append(nn.ModuleDict({
                'attn_norm': RMSNorm(d_model),
                'attn': GroupedQueryAttention(d_model, n_heads, n_kv_heads, d_model // n_heads),
                'moe_norm': RMSNorm(d_model),
                'moe': SparseMoELayer(d_model, d_ff, num_experts, top_k)
            }))

        self.final_norm = RMSNorm(d_model)
        self.reward_head = nn.Linear(d_model, 1)

    def forward(self, input_ids):
        B, S = input_ids.shape
        x = self.drop(self.tok_embeddings(input_ids))
        cos, sin = self.rope(seq_len=S)

        total_aux_loss = 0.0
        for layer in self.layers:
            x = x + self.drop(layer['attn'](layer['attn_norm'](x), cos, sin))
            moe_out, aux_loss = layer['moe'](layer['moe_norm'](x))
            x = x + self.drop(moe_out)
            total_aux_loss = total_aux_loss + aux_loss

        x = self.final_norm(x)
        logits = self.reward_head(x).squeeze(-1)
        return logits, total_aux_loss

## 4. INITIALIZE MODEL & PRINT PARAMETER METRICS

In [5]:
seed_everything(42)

model = AegisStep28M(
    vocab_size=len(tokenizer),
    d_model=288, n_layers=6, n_heads=8, n_kv_heads=2,
    d_ff=576, num_experts=4, top_k=2
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n" + "="*55)
print("     AEGISSTEP-28M MODEL ARCHITECTURE SPECIFICATIONS")
print("="*55)
print(f"Model Name:            AegisStep-28M")
print(f"Total Parameters:      {total_params:,} ({total_params/1e6:.2f}M)")
print(f"Trainable Parameters:  {trainable_params:,}")
print(f"Routing Mechanism:     Top-2 Gated Sparse Mixture-of-Experts")
print(f"Total Experts:         4 Experts per Layer (6 Layers)")
print(f"Attention Type:        Grouped-Query Attention (GQA)")
print(f"Position Embeddings:   Rotary Positional Embeddings (RoPE)")
print(f"Activation Function:   SwiGLU")
print("="*55 + "\n")


     AEGISSTEP-28M MODEL ARCHITECTURE SPECIFICATIONS
Model Name:            AegisStep-28M
Total Parameters:      27,673,921 (27.67M)
Trainable Parameters:  27,673,921
Routing Mechanism:     Top-2 Gated Sparse Mixture-of-Experts
Total Experts:         4 Experts per Layer (6 Layers)
Attention Type:        Grouped-Query Attention (GQA)
Position Embeddings:   Rotary Positional Embeddings (RoPE)
Activation Function:   SwiGLU



## 5. TRAINING (hash-based splits, LR schedule, val-loss checkpointing)

In [6]:
MAX_LEN = 512
train_dataset = MathShepherdDataset(tokenizer, "train", max_items=150000, max_seq_len=MAX_LEN)
val_dataset   = MathShepherdDataset(tokenizer, "val",   max_items=10000,  max_seq_len=MAX_LEN)
test_dataset  = MathShepherdDataset(tokenizer, "test",  max_items=10000,  max_seq_len=MAX_LEN)

BATCH = 32   # use 16 if you hit out-of-memory
train_loader = DataLoader(train_dataset, batch_size=BATCH, shuffle=True,  num_workers=2,
                          pin_memory=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH, shuffle=False, num_workers=2,
                          pin_memory=True, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH, shuffle=False, num_workers=2,
                          pin_memory=True, collate_fn=collate_fn)

EPOCHS = 3
PATIENCE = 2
AUX_LOSS_COEFF = 0.0001

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-4,
    total_steps=EPOCHS * len(train_loader),
    pct_start=0.05, anneal_strategy='cos'
)
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
bce_loss_fn = nn.BCEWithLogitsLoss(reduction='none')

best_val_loss, bad_epochs = float("inf"), 0


@torch.no_grad()
def collect(loader):
    model.eval()
    probs_all, tgts_all, chains = [], [], []
    loss_sum, n = 0.0, 0
    for batch in loader:
        ids = batch["input_ids"].to(DEVICE)
        lab = batch["step_labels"].to(DEVICE)
        logits, _ = model(ids)
        logits = logits.float()
        for b in range(ids.size(0)):
            m = lab[b] != -100
            if m.sum() > 0:
                p = torch.sigmoid(logits[b][m]).cpu().numpy()
                t = lab[b][m].cpu().numpy()
                probs_all.extend(p)
                tgts_all.extend(t)
                chains.append((p, t))
        m_all = lab != -100
        if m_all.sum() > 0:
            loss_sum += bce_loss_fn(logits[m_all], lab[m_all]).mean().item()
            n += 1
    return loss_sum / max(n, 1), np.array(probs_all), np.array(tgts_all), chains


def metrics_at(probs, tgts, thr):
    preds = (probs > thr).astype(float)
    p, r, f, _ = precision_recall_fscore_support(tgts, preds, average='binary', zero_division=0)
    return accuracy_score(tgts, preds), p, r, f

def auroc(probs, tgts):
    return roc_auc_score(tgts, probs) if len(np.unique(tgts)) > 1 else float("nan")


print("[*] Training AegisStep-30M on Math-Shepherd Reasoning Chains...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, total_prm, total_aux = 0.0, 0.0, 0.0
    start_time = time.time()

    for batch in train_loader:
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        labels = batch["step_labels"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            logits, aux_loss = model(input_ids)
            mask = (labels != -100)
            if mask.sum() > 0:
                prm_loss = bce_loss_fn(logits[mask].float(), labels[mask]).mean()
            else:
                prm_loss = torch.tensor(0.0, device=DEVICE)
            loss = prm_loss + AUX_LOSS_COEFF * aux_loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()
        total_prm += prm_loss.item()
        total_aux += aux_loss.item()

    n_batches = len(train_loader)
    val_loss, vp, vt, _ = collect(val_loader)
    val_acc, _, _, val_f1 = metrics_at(vp, vt, 0.5)
    val_auc = auroc(vp, vt)
    print(f"Epoch {epoch:02d}/{EPOCHS:02d} | Train PRM: {total_prm/n_batches:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}% | "
          f"Val F1: {val_f1*100:.2f}% | Val AUROC: {val_auc:.4f} | "
          f"Time: {time.time()-start_time:.1f}s")

    if val_loss < best_val_loss:
        best_val_loss, bad_epochs = val_loss, 0
        torch.save(model.state_dict(), "best_aegisstep.pt")
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"[*] Early stopping at epoch {epoch}.")
            break

model.load_state_dict(torch.load("best_aegisstep.pt", map_location=DEVICE))



[*] Building 'train' split...
[*] Loading peiyi9979/Math-Shepherd (one time)...


README.md: 0.00B [00:00, ?B/s]

math-shepherd.jsonl:   0%|          | 0.00/793M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/444655 [00:00<?, ? examples/s]

[+] train: 150000 chains | positive-step rate: 49.0%
[*] Building 'val' split...
[+] val: 10000 chains | positive-step rate: 48.1%
[*] Building 'test' split...
[+] test: 10000 chains | positive-step rate: 48.5%
[*] Training AegisStep-30M on Math-Shepherd Reasoning Chains...
Epoch 01/03 | Train PRM: 0.5889 | Val Loss: 0.5933 | Val Acc: 68.14% | Val F1: 70.11% | Val AUROC: 0.7556 | Time: 1809.1s
Epoch 02/03 | Train PRM: 0.5244 | Val Loss: 0.5901 | Val Acc: 69.58% | Val F1: 69.26% | Val AUROC: 0.7636 | Time: 1808.4s
Epoch 03/03 | Train PRM: 0.4723 | Val Loss: 0.6173 | Val Acc: 69.33% | Val F1: 70.21% | Val AUROC: 0.7567 | Time: 1806.2s


<All keys matched successfully>

## 6. THRESHOLD TUNING (val) + FINAL EVALUATION (held-out test)

In [7]:
_, vp, vt, _ = collect(val_loader)
best_thr, best_thr_f1 = 0.5, 0.0
for thr in np.arange(0.1, 0.91, 0.02):
    _, _, _, f = metrics_at(vp, vt, thr)
    if f > best_thr_f1:
        best_thr, best_thr_f1 = float(thr), f
print(f"[*] Best threshold (val): {best_thr:.2f} | Val F1: {best_thr_f1*100:.2f}%")

print("\n[*] Running Final Evaluation on Held-Out Test Split...")
_, tp, tt, test_chains = collect(test_loader)
accuracy, precision, recall, f1 = metrics_at(tp, tt, best_thr)
test_auc = auroc(tp, tt)

pos_rate = float(tt.mean())
majority_baseline = max(pos_rate, 1 - pos_rate)
trivial_f1 = 2 * pos_rate / (1 + pos_rate)
acc05, _, _, f05 = metrics_at(tp, tt, 0.5)
bal_acc = balanced_accuracy_score(tt, (tp > 0.5).astype(float))

chain_correct = sum(np.array_equal((p > best_thr).astype(float), t) for p, t in test_chains)
chain_acc = chain_correct / len(test_chains) if test_chains else 0.0

print("\n" + "="*55)
print("       AEGISSTEP-30M BENCHMARK PERFORMANCE METRICS")
print("="*55)
print(f"Test Positive-Step Rate:           {pos_rate * 100:.2f}%")
print(f"Majority-Class Baseline Accuracy:  {majority_baseline * 100:.2f}%")
print(f"Trivial 'always correct' F1:       {trivial_f1 * 100:.2f}%")
print(f"Decision Threshold (val-tuned):    {best_thr:.2f}")
print(f"Step-Level Verification Accuracy:  {accuracy * 100:.2f}%")
print(f"Step-Level Precision:              {precision * 100:.2f}%")
print(f"Step-Level Recall:                 {recall * 100:.2f}%")
print(f"Step-Level F1-Score:               {f1 * 100:.2f}%")
print(f"Step-Level AUROC:                  {test_auc:.4f}")
print(f"Accuracy @0.5 / Balanced Acc @0.5: {acc05*100:.2f}% / {bal_acc*100:.2f}%")
print(f"Full Chain Verification Accuracy:  {chain_acc * 100:.2f}%")
print("="*55 + "\n")

[*] Best threshold (val): 0.30 | Val F1: 72.47%

[*] Running Final Evaluation on Held-Out Test Split...

       AEGISSTEP-30M BENCHMARK PERFORMANCE METRICS
Test Positive-Step Rate:           50.54%
Majority-Class Baseline Accuracy:  50.54%
Trivial 'always correct' F1:       67.14%
Decision Threshold (val-tuned):    0.30
Step-Level Verification Accuracy:  67.87%
Step-Level Precision:              63.70%
Step-Level Recall:                 84.67%
Step-Level F1-Score:               72.70%
Step-Level AUROC:                  0.7727
Accuracy @0.5 / Balanced Acc @0.5: 69.97% / 69.99%
Full Chain Verification Accuracy:  39.97%

